# Weibin batch preview with saved Pt ROI

Run the notebook top to bottom.

1. The settings cell loads `saved_rois/Pt_L3_roi.json`.
2. The ROI editor is optional; when enabled, it uses a PyQtGraph selector by default.
3. Every usable non-flatfield `UFIS*count_multiple` data file is paired with `FIXED_FLATFIELD_FILE` when set, otherwise the nearest `flatfield_lambda-count_multiple` HDF.
4. Preview HTML files are exported with the saved ROI highlighted in cyan.

In [ ]:
import json
import os
import re
import sys
import types
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from urllib.parse import quote
import html


def find_repo_root(start=None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'Python_codes' / 'Dispersive_XAS').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


REPO_ROOT = find_repo_root()

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / '20260604_run_preview.ipynb').exists():
    for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (candidate / '20260604_run_preview.ipynb').exists():
            NOTEBOOK_DIR = candidate
            break

DATA_DIR = NOTEBOOK_DIR / 'data' if (NOTEBOOK_DIR / 'data').is_dir() else NOTEBOOK_DIR
COMPAT_DATA_DIR = NOTEBOOK_DIR / 'derived_legacy_h5'
SCAN_TXT = NOTEBOOK_DIR / 'scan.txt'
EPICS_UNIX_OFFSET = 631_152_000

SAFE_PLUGIN_DIR = NOTEBOOK_DIR / '.hdf5_plugin'
SAFE_PLUGIN_DIR.mkdir(exist_ok=True)
if not Path(os.environ.get('HDF5_PLUGIN_PATH', '')).exists():
    os.environ['HDF5_PLUGIN_PATH'] = str(SAFE_PLUGIN_DIR)

try:
    import hdf5plugin  # noqa: F401
except Exception:
    pass

import h5py
import matplotlib.pyplot as plt
import numpy as np

if str(REPO_ROOT / 'Python_codes') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'Python_codes'))

# Dispersive_XAS imports skimage during package initialization, but this
# preview path only uses ROI and batch-preview helpers. Provide a minimal
# fallback so a kernel without skimage can still run these cells.
try:
    import skimage  # noqa: F401
except ModuleNotFoundError:
    sys.modules.setdefault('skimage', types.ModuleType('skimage'))

try:
    from Dispersive_XAS import (
        infer_tilted_band_roi_from_paths,
        load_roi_json,
        make_tilted_band_roi,
        normalize_roi_spec,
        plot_spectra_in_chunks,
        preview_spectra_html,
        roi_boundary_rows,
        roi_weighted_column_mean,
        save_roi_json,
        select_tilted_band_roi,
        tilted_band_controls_from_roi,
    )
except ImportError:
    import importlib
    import Dispersive_XAS.core.roi as dxas_core_roi
    import Dispersive_XAS.web.roi as dxas_web_roi
    importlib.reload(dxas_core_roi)
    importlib.reload(dxas_web_roi)
    from Dispersive_XAS import (
        infer_tilted_band_roi_from_paths,
        make_tilted_band_roi,
        normalize_roi_spec,
        plot_spectra_in_chunks,
        preview_spectra_html,
        roi_boundary_rows,
        roi_weighted_column_mean,
        tilted_band_controls_from_roi,
    )
    from Dispersive_XAS.core.roi import load_roi_json, save_roi_json
    from Dispersive_XAS.web.roi import select_tilted_band_roi

In [ ]:

SCAN_LINE_RE = re.compile(
    r'^\s*(?P<idx>-?\d+)\s+'
    r'(?P<scan_id>\d+)\s+'
    r'(?P<time>\S+)\s+'
    r'(?P<sample_name>.+?)\s+'
    r'(?P<uid>[0-9a-f-]{36})\s*$'
)
DATE_DIR_RE = re.compile(r'^\d{4}-\d{2}-\d{2}$')


def decode_text(value):
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='replace')
    if isinstance(value, np.generic):
        return value.item()
    return value


def parse_jsonish(value):
    value = decode_text(value)
    if isinstance(value, str):
        stripped = value.strip()
        if stripped and stripped[0] in '[{':
            try:
                return json.loads(stripped)
            except Exception:
                return value
    return value


def sanitize_name(value):
    return re.sub(r'[^0-9A-Za-z._-]+', '_', str(value)).strip('_')


def exported_h5_paths():
    roots = [path for path in sorted(DATA_DIR.iterdir()) if path.is_dir() and DATE_DIR_RE.match(path.name)]
    if not roots:
        roots = [DATA_DIR]
    seen = set()
    for root in roots:
        for pattern in ('*.h5', '*.hdf', '*.H5', '*.HDF'):
            for path in sorted(root.glob(pattern)):
                if path not in seen:
                    seen.add(path)
                    yield path


def default_entry_key(h5_file):
    entry_key = decode_text(h5_file.attrs.get('default'))
    if entry_key and entry_key in h5_file:
        return entry_key
    if 'entry' in h5_file:
        return 'entry'
    for key, value in h5_file.items():
        if isinstance(value, h5py.Group) and 'instrument/bluesky/metadata' in value:
            return key
    return None


def h5_metadata_value(h5_file, key, default=None):
    entry_key = default_entry_key(h5_file)
    if not entry_key:
        return default
    metadata_path = f'{entry_key}/instrument/bluesky/metadata'
    if metadata_path not in h5_file:
        return default
    metadata = h5_file[metadata_path]
    if key not in metadata:
        return default
    return parse_jsonish(metadata[key][()])


def lambda_dataset_path(h5_file):
    for key in ('entry/data/data', 'entry/instrument/detector/data'):
        if key in h5_file:
            return h5_file[key].name
    entry_key = default_entry_key(h5_file)
    if entry_key:
        for key in (
            f'{entry_key}/instrument/bluesky/streams/primary/lambda_250k/value',
            f'{entry_key}/instrument/bluesky/streams/primary/lambda/value',
        ):
            if key in h5_file:
                return h5_file[key].name
    raise KeyError('Could not find a lambda detector dataset in the HDF5 file.')


def detector_dataset(h5_file):
    for key in ('entry/data/data', 'entry/instrument/detector/data'):
        if key in h5_file:
            return h5_file[key]
    return h5_file[lambda_dataset_path(h5_file)]


def inspect_export(path):
    with h5py.File(path, 'r') as h5_file:
        dataset = detector_dataset(h5_file)
        start_time_unix = h5_metadata_value(h5_file, 'start.time')
        if start_time_unix is not None:
            scan_time = datetime.fromtimestamp(float(start_time_unix))
        else:
            filename_match = re.match(r'(?P<stamp>\d{12})-', path.name)
            scan_time = datetime.strptime(filename_match.group('stamp'), '%Y%m%d%H%M') if filename_match else None
        uid = decode_text(h5_metadata_value(h5_file, 'start.uid', path.stem))
        scan_id = int(h5_metadata_value(h5_file, 'start.scan_id', -1))
        sample_name = decode_text(h5_metadata_value(h5_file, 'start.sample_name', '')) or path.stem
        plan_name = decode_text(h5_metadata_value(h5_file, 'start.plan_name', ''))
        shape = tuple(int(v) for v in dataset.shape)
    return {
        'path': path,
        'name': path.name,
        'uid': uid,
        'scan_id': scan_id,
        'scan_time': scan_time,
        'time': scan_time.strftime('%Y-%m-%d:%H-%M') if scan_time else '',
        'sample_name': sample_name,
        'plan_name': plan_name,
        'shape': shape,
        'nframes': shape[0] if len(shape) >= 3 else 1,
    }


def ensure_symlink(alias_path, target_path):
    if alias_path.exists() or alias_path.is_symlink():
        return alias_path
    try:
        alias_path.symlink_to(target_path.name)
        return alias_path
    except OSError:
        return target_path


@lru_cache(maxsize=1)
def scan_entries():
    if SCAN_TXT.exists():
        entries = []
        for line in SCAN_TXT.read_text(encoding='utf-8').splitlines():
            match = SCAN_LINE_RE.match(line)
            if match is None:
                continue
            entry = match.groupdict()
            entry['scan_id'] = int(entry['scan_id'])
            entry['scan_time'] = datetime.strptime(entry['time'], '%Y-%m-%d:%H-%M')
            entry['path'] = None
            entries.append(entry)
        return entries

    entries = []
    for path in exported_h5_paths():
        try:
            entries.append(inspect_export(path))
        except Exception:
            continue
    entries.sort(key=lambda entry: entry['scan_time'] or datetime.min)
    return entries


def scan_entry_by_date_id(date_text, scan_id):
    date_text = str(date_text).strip()
    for entry in scan_entries():
        if entry['scan_id'] == int(scan_id) and entry['time'].startswith(date_text):
            return entry
    raise KeyError(f'No HDF5 metadata entry matched date={date_text!r}, scan_id={scan_id!r}.')


@lru_cache(maxsize=1)
def exported_entries():
    return tuple(scan_entries())


def resolve_uid_path(scan_entry):
    if scan_entry.get('path') is not None:
        return Path(scan_entry['path'])
    uid_path = DATA_DIR / f"{scan_entry['uid']}.h5"
    if uid_path.exists() or uid_path.is_symlink():
        return uid_path

    best_export = min(
        exported_entries(),
        key=lambda entry: abs((scan_entry['scan_time'] - entry['scan_time']).total_seconds()),
    )['path']
    return ensure_symlink(uid_path, best_export)


def preview_stem(scan_entry):
    return f"{scan_entry['scan_time'].strftime('%Y%m%d_%H%M')}{sanitize_name(scan_entry['sample_name'])}"


def find_sample_entry(sample_name, date_text=None, plan_name='count_multiple'):
    sample_key = str(sample_name).strip().lower()
    date_key = str(date_text).strip() if date_text is not None else None
    matches = []
    for entry in scan_entries():
        if date_key is not None and not entry['time'].startswith(date_key):
            continue
        if plan_name is not None and entry.get('plan_name') != plan_name:
            continue
        if str(entry['sample_name']).strip().lower() == sample_key:
            matches.append(entry)
    if not matches:
        raise KeyError(f'No exported HDF5 file matched sample={sample_name!r}, date={date_text!r}, plan={plan_name!r}.')
    return sorted(matches, key=lambda entry: entry['scan_time'])[0]


def find_nearest_flatfield(data_entry, flatfield_sample_name='flatfield_lambda', prefer='after'):
    flat_key = str(flatfield_sample_name).strip().lower()
    same_day = [
        entry for entry in scan_entries()
        if entry['scan_time'] is not None
        and data_entry['scan_time'] is not None
        and entry['scan_time'].date() == data_entry['scan_time'].date()
        and entry.get('plan_name') == 'count_multiple'
        and flat_key in str(entry['sample_name']).strip().lower()
    ]
    if not same_day:
        raise KeyError(f'No flatfield entry matched {flatfield_sample_name!r} on {data_entry["time"][:10]}.')
    if prefer == 'after':
        after = [entry for entry in same_day if entry['scan_time'] >= data_entry['scan_time']]
        if after:
            return min(after, key=lambda entry: entry['scan_time'] - data_entry['scan_time'])
    if prefer == 'before':
        before = [entry for entry in same_day if entry['scan_time'] <= data_entry['scan_time']]
        if before:
            return min(before, key=lambda entry: data_entry['scan_time'] - entry['scan_time'])
    return min(same_day, key=lambda entry: abs((entry['scan_time'] - data_entry['scan_time']).total_seconds()))


def ensure_legacy_lambda_view(source_path, view_path):
    source_path = Path(source_path).resolve()
    view_path = Path(view_path).resolve()
    view_path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(source_path, 'r') as source_file:
        target_dataset = lambda_dataset_path(source_file)
    if view_path.exists() or view_path.is_symlink():
        view_path.unlink()
    with h5py.File(view_path, 'w') as view_file:
        data_group = view_file.create_group('entry').create_group('data')
        data_group['data'] = h5py.ExternalLink(str(source_path), target_dataset)
        view_file.attrs['source_file'] = str(source_path)
        view_file.attrs['source_dataset'] = target_dataset
    return view_path


def infer_norm_range(data_path, flat_path, row_range, roi, window, sample_frames):
    with h5py.File(flat_path, 'r') as f:
        flat_avg = np.asarray(detector_dataset(f)[:], dtype=np.float32).mean(axis=0)

    with h5py.File(data_path, 'r') as f:
        data = np.asarray(detector_dataset(f)[:sample_frames], dtype=np.float32)

    with np.errstate(divide='ignore', invalid='ignore'):
        mux = np.log(np.clip(flat_avg[None, :, :], 1e-6, None) / np.clip(data, 1e-6, None))
    mux[~np.isfinite(mux)] = 0.0
    mean_spec = roi_weighted_column_mean(mux, row_range=row_range, roi=roi).mean(axis=0)

    width = int(mean_spec.shape[0])
    if width <= 0:
        raise RuntimeError('Could not infer a normalization window from an empty spectrum.')
    if window >= width:
        return 0, width

    best_span = -np.inf
    best_start = 0
    for start in range(0, width - window + 1):
        segment = mean_spec[start:start + window]
        span = float(np.nanmax(segment) - np.nanmin(segment))
        if span > best_span:
            best_span = span
            best_start = start

    return int(best_start), int(best_start + window)


def load_mux_preview_frame(data_path, flat_path, frame_index):
    with h5py.File(flat_path, 'r') as f:
        flat_avg = np.asarray(detector_dataset(f)[:], dtype=np.float32).mean(axis=0)

    with h5py.File(data_path, 'r') as f:
        data_ds = detector_dataset(f)
        frame_index = max(0, min(int(frame_index), int(data_ds.shape[0]) - 1))
        data_frame = np.asarray(data_ds[frame_index], dtype=np.float32)

    with np.errstate(divide='ignore', invalid='ignore'):
        mux_frame = np.log(np.clip(flat_avg, 1e-6, None) / np.clip(data_frame, 1e-6, None))
    mux_frame[~np.isfinite(mux_frame)] = 0.0
    return mux_frame, frame_index


def _qt_exec(obj):
    exec_fn = getattr(obj, 'exec', None) or getattr(obj, 'exec_', None)
    if exec_fn is None:
        raise RuntimeError('Qt object does not expose exec/exec_.')
    return exec_fn()


def _display_image_and_limits(img, q_low=1.0, q_high=99.0):
    arr = np.asarray(img, dtype=float)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr, dtype=float), 0.0, 1.0
    lo, hi = np.nanpercentile(finite, [q_low, q_high])
    if not np.isfinite(lo) or not np.isfinite(hi):
        lo = float(np.nanmin(finite))
        hi = float(np.nanmax(finite))
    if hi <= lo:
        center = float(finite[0])
        span = max(1.0, abs(center) * 1e-3)
        lo = center - span
        hi = center + span
    disp = np.array(arr, copy=True, dtype=float)
    disp[~np.isfinite(disp)] = lo
    return disp, float(lo), float(hi)


class PyQtGraphTiltedBandSelector:
    """PyQtGraph tilted-band ROI selector for one representative mux image."""

    def __init__(self, image, initial_roi=None, title='', save_path=None):
        self.image = np.asarray(image, dtype=float)
        if self.image.ndim != 2:
            raise ValueError('PyQtGraphTiltedBandSelector expects a 2-D image.')
        self.title = title or 'DXAS tilted-band ROI selector'
        self.save_path = None if save_path is None else Path(save_path).expanduser().resolve()
        self.roi = self._normalize_initial_roi(initial_roi)
        self._window = None
        self._image_plot = None
        self._histogram_lut = None
        self._center_roi = None
        self._top_curve = None
        self._bottom_curve = None
        self._spectrum_curve = None
        self._half_width_spin = None
        self._height_spin = None
        self._path_edit = None
        self._status_label = None
        self._syncing = False

    def _normalize_initial_roi(self, initial_roi):
        h, _w = self.image.shape
        if initial_roi is None:
            center = 0.5 * max(0.0, float(h - 1))
            return make_tilted_band_roi(
                self.image.shape,
                left_center_row=center,
                right_center_row=center,
                half_width=max(4.0, float(h) * 0.06),
            )
        return normalize_roi_spec(self.image.shape, roi=initial_roi)

    def _summary_text(self):
        left, right, half_width = tilted_band_controls_from_roi(self.image.shape, roi=self.roi)
        height = 2.0 * float(half_width)
        slope = float(self.roi.get('slope_per_col', 0.0))
        bounds = list(self.roi.get('row_bounds', []))
        return (
            f'ROI: left={left:.1f}, right={right:.1f}, '
            f'height={height:.1f} px, half_width={half_width:.1f}, '
            f'slope={slope:.4f}, row_bounds={bounds}'
        )

    def _line_endpoints(self):
        h, w = self.image.shape
        if self._center_roi is None:
            left, right, _half_width = tilted_band_controls_from_roi(self.image.shape, roi=self.roi)
            return (0.0, float(left)), (float(max(1, w - 1)), float(right))

        try:
            view_box = self._image_plot.getViewBox() if self._image_plot is not None else None
            points = []
            for _handle, scene_pos in self._center_roi.getSceneHandlePositions()[:2]:
                pos = view_box.mapSceneToView(scene_pos) if view_box is not None else scene_pos
                points.append((float(pos.x()), float(pos.y())))
            if len(points) == 2:
                return points[0], points[1]
        except Exception:
            pass

        try:
            points = []
            for _handle, local_pos in self._center_roi.getLocalHandlePositions()[:2]:
                pos = self._center_roi.mapToParent(local_pos)
                points.append((float(pos.x()), float(pos.y())))
            if len(points) == 2:
                return points[0], points[1]
        except Exception:
            pass

        left, right, _half_width = tilted_band_controls_from_roi(self.image.shape, roi=self.roi)
        return (0.0, float(left)), (float(max(1, w - 1)), float(right))

    def _sync_roi_from_graphics(self):
        if self._syncing:
            return
        h, w = self.image.shape
        (x0, y0), (x1, y1) = self._line_endpoints()
        if x1 < x0:
            x0, y0, x1, y1 = x1, y1, x0, y0
        if abs(x1 - x0) <= 1e-9:
            left = float(y0)
            right = float(y1)
        else:
            slope = (float(y1) - float(y0)) / (float(x1) - float(x0))
            left = float(y0) + slope * (0.0 - float(x0))
            right = float(y0) + slope * (float(w - 1) - float(x0))
        max_row = max(0.0, float(h - 1))
        left = float(np.clip(left, 0.0, max_row))
        right = float(np.clip(right, 0.0, max_row))
        half_width = float(self.roi.get('half_width', 1.0))
        if self._height_spin is not None:
            half_width = max(0.5, 0.5 * float(self._height_spin.value()))
        elif self._half_width_spin is not None:
            half_width = float(self._half_width_spin.value())
        self.roi = make_tilted_band_roi(self.image.shape, left, right, half_width)

    def _refresh(self):
        self._sync_roi_from_graphics()
        cols, top, bottom = roi_boundary_rows(self.image.shape, roi=self.roi)
        spec = roi_weighted_column_mean(self.image, roi=self.roi)
        if self._height_spin is not None:
            self._syncing = True
            self._height_spin.setValue(float(2.0 * float(self.roi.get('half_width', 1.0))))
            self._syncing = False
        if self._top_curve is not None:
            self._top_curve.setData(cols, top)
        if self._bottom_curve is not None:
            self._bottom_curve.setData(cols, bottom)
        if self._spectrum_curve is not None:
            self._spectrum_curve.setData(np.arange(self.image.shape[1], dtype=float), spec)
        if self._status_label is not None:
            self._status_label.setText(self._summary_text())

    def _on_graphics_changed(self):
        if self._syncing:
            return
        self._refresh()

    def _on_half_width_changed(self, _value):
        if self._syncing:
            return
        self._refresh()

    def _on_height_changed(self, _value):
        if self._syncing:
            return
        self._refresh()

    def get_spec(self):
        self._sync_roi_from_graphics()
        return normalize_roi_spec(self.image.shape, roi=self.roi)

    def save(self, path=None, metadata=None):
        out_path = Path(path or self.save_path).expanduser() if path or self.save_path else None
        if out_path is None:
            raise ValueError('No ROI JSON path was provided.')
        payload_meta = {'shape': [int(self.image.shape[0]), int(self.image.shape[1])]}
        if metadata:
            payload_meta.update(metadata)
        return save_roi_json(out_path, self.get_spec(), metadata=payload_meta)

    def _save_from_gui(self, close_after=False):
        try:
            path_text = self._path_edit.text().strip() if self._path_edit is not None else ''
            saved = self.save(path=path_text or None)
            if self._status_label is not None:
                self._status_label.setText(f'Saved ROI JSON: {saved}')
            if close_after and self._window is not None:
                self._window.close()
        except Exception as exc:
            if self._status_label is not None:
                self._status_label.setText(f'Save failed: {exc}')

    def launch(self, show=True, block=True):
        if not show:
            return self
        try:
            import pyqtgraph as pg
            from pyqtgraph.Qt import QtCore, QtWidgets
        except Exception as exc:
            raise ImportError(
                'PyQtGraph ROI selector requires pyqtgraph and a Qt binding. '
                'Install with: pip install pyqtgraph PyQt6'
            ) from exc

        app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])
        try:
            pg.setConfigOptions(imageAxisOrder='row-major')
        except Exception:
            pass

        h, w = self.image.shape
        left, right, half_width = tilted_band_controls_from_roi(self.image.shape, roi=self.roi)
        disp, zmin, zmax = _display_image_and_limits(self.image)

        window = QtWidgets.QMainWindow()
        window.setWindowTitle(self.title)
        attr = getattr(QtCore.Qt, 'WA_DeleteOnClose', None)
        if attr is None and hasattr(QtCore.Qt, 'WidgetAttribute'):
            attr = QtCore.Qt.WidgetAttribute.WA_DeleteOnClose
        if attr is not None:
            window.setAttribute(attr, True)

        central = QtWidgets.QWidget()
        layout = QtWidgets.QVBoxLayout(central)
        graphics = pg.GraphicsLayoutWidget()
        layout.addWidget(graphics, 1)

        image_plot = graphics.addPlot(row=0, col=0, title='Mux image with draggable ROI center line')
        image_plot.setLabel('bottom', 'Detector column')
        image_plot.setLabel('left', 'Detector row')
        image_plot.invertY(True)
        try:
            image_item = pg.ImageItem(axisOrder='row-major')
        except TypeError:
            image_item = pg.ImageItem()
        image_item.setImage(disp, levels=(zmin, zmax), autoLevels=False)
        try:
            image_item.setRect(QtCore.QRectF(0, 0, float(w), float(h)))
        except Exception:
            pass
        image_plot.addItem(image_item)
        histogram_lut = pg.HistogramLUTItem(image=image_item)
        histogram_lut.setLevels(zmin, zmax)
        try:
            histogram_lut.gradient.loadPreset('magma')
        except Exception:
            pass
        graphics.addItem(histogram_lut, row=0, col=1)
        try:
            graphics.ci.layout.setColumnMaximumWidth(1, 120)
        except Exception:
            pass

        spectrum_plot = graphics.addPlot(row=1, col=0, title='ROI vertical average spectrum')
        spectrum_plot.setLabel('bottom', 'Detector column')
        spectrum_plot.setLabel('left', 'Mean mux')

        top_curve = image_plot.plot([], [], pen=pg.mkPen((0, 255, 255), width=2))
        bottom_curve = image_plot.plot([], [], pen=pg.mkPen((0, 255, 255), width=2))
        try:
            fill = pg.FillBetweenItem(top_curve, bottom_curve, brush=pg.mkBrush(0, 255, 255, 45))
            image_plot.addItem(fill)
        except Exception:
            pass
        spectrum_curve = spectrum_plot.plot([], [], pen=pg.mkPen((60, 140, 255), width=2))
        center_roi = pg.LineSegmentROI([[0.0, left], [float(max(1, w - 1)), right]], pen=pg.mkPen('w', width=2))
        image_plot.addItem(center_roi)

        controls = QtWidgets.QHBoxLayout()
        controls.addWidget(QtWidgets.QLabel('ROI height'))
        height_spin = QtWidgets.QDoubleSpinBox()
        height_spin.setRange(1.0, max(2.0, float(h)))
        height_spin.setSingleStep(1.0)
        height_spin.setDecimals(1)
        height_spin.setValue(float(2.0 * half_width))
        height_spin.setToolTip('Full vertical height of the cyan ROI band in detector pixels')
        controls.addWidget(height_spin)
        controls.addWidget(QtWidgets.QLabel('ROI JSON'))
        path_edit = QtWidgets.QLineEdit('' if self.save_path is None else str(self.save_path))
        controls.addWidget(path_edit, 1)
        save_button = QtWidgets.QPushButton('Save ROI JSON')
        save_close_button = QtWidgets.QPushButton('Save and close')
        close_button = QtWidgets.QPushButton('Close')
        controls.addWidget(save_button)
        controls.addWidget(save_close_button)
        controls.addWidget(close_button)
        layout.addLayout(controls)

        status_label = QtWidgets.QLabel(self._summary_text())
        layout.addWidget(status_label)
        help_label = QtWidgets.QLabel('Drag the colorbar level handles to adjust vmin/vmax; drag the white line to adjust ROI center/slope; use ROI height to widen or shrink the band.')
        layout.addWidget(help_label)
        window.setCentralWidget(central)
        window.resize(1200, 900)

        self._window = window
        self._image_plot = image_plot
        self._histogram_lut = histogram_lut
        self._center_roi = center_roi
        self._top_curve = top_curve
        self._bottom_curve = bottom_curve
        self._spectrum_curve = spectrum_curve
        self._height_spin = height_spin
        self._path_edit = path_edit
        self._status_label = status_label

        center_roi.sigRegionChanged.connect(self._on_graphics_changed)
        height_spin.valueChanged.connect(self._on_height_changed)
        save_button.clicked.connect(lambda: self._save_from_gui(close_after=False))
        save_close_button.clicked.connect(lambda: self._save_from_gui(close_after=True))
        close_button.clicked.connect(window.close)

        self._refresh()
        window.show()
        window.raise_()
        if block:
            loop = QtCore.QEventLoop()
            window.destroyed.connect(loop.quit)
            _qt_exec(loop)
        return self


def select_tilted_band_roi_pyqtgraph(image, initial_roi=None, title='', save_path=None, show=True, block=True):
    editor = PyQtGraphTiltedBandSelector(image, initial_roi=initial_roi, title=title, save_path=save_path)
    return editor.launch(show=show, block=block)


def select_roi_for_processing(data_path, flat_path, initial_roi, roi_json_path, data_entry):
    mux_preview, preview_frame_index = load_mux_preview_frame(data_path, flat_path, ROI_PREVIEW_FRAME)
    backend = str(ROI_EDITOR_BACKEND).strip().lower()
    title = f"ROI selector - {data_entry['name']} frame {preview_frame_index}"
    if backend in {'pyqtgraph', 'qt', 'pg'}:
        editor = select_tilted_band_roi_pyqtgraph(
            mux_preview,
            initial_roi=initial_roi,
            title=title,
            save_path=roi_json_path,
            show=True,
            block=ROI_EDITOR_BLOCK,
        )
    elif backend in {'html', 'plotly', 'widgets'}:
        editor = select_tilted_band_roi(
            mux_preview,
            initial_roi=initial_roi,
            title=title,
            save_path=roi_json_path,
            show=True,
        )
    else:
        raise ValueError(f'Unsupported ROI_EDITOR_BACKEND: {ROI_EDITOR_BACKEND!r}')
    return editor.get_spec()


def show_roi_preview(data_path, flat_path, row_range, roi, frame_index):
    mux_preview, preview_frame_index = load_mux_preview_frame(data_path, flat_path, frame_index)
    cols, top_rows, bottom_rows = roi_boundary_rows(mux_preview.shape, row_range=row_range, roi=roi)
    vmin, vmax = np.percentile(mux_preview, [1, 99])
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(
        mux_preview,
        cmap='magma',
        aspect='auto',
        origin='lower',
        vmin=vmin,
        vmax=vmax,
    )
    ax.fill_between(cols, top_rows, bottom_rows, color='cyan', alpha=0.18)
    ax.plot(cols, top_rows, color='cyan', linewidth=1.5)
    ax.plot(cols, bottom_rows, color='cyan', linewidth=1.5)
    ax.set_title(f'mux frame {preview_frame_index} with ROI')
    ax.set_xlabel('detector column')
    ax.set_ylabel('detector row')
    fig.colorbar(im, ax=ax, label='mu')
    plt.show()


## Settings

Edit only this cell for day-to-day use.


In [ ]:
COUNT_MULTIPLE_GLOB = '*UFIS*count_multiple*.hdf'
FLATFIELD_NAME_TOKEN = 'flatfield_lambda-count_multiple'
FIXED_FLATFIELD_FILE = '202606041408-flatfield_lambda-count_multiple-8891a181.hdf'
DATA_EXCLUDE_TOKENS = ('flatfield',)
ONLY_DATA_FILES = None  # process all matching UFIS non-flatfield count_multiple data files
FLATFIELD_PREFERENCE = 'nearest'  # used only when FIXED_FLATFIELD_FILE is None

ROI_JSON_NAME = None
ROI_JSON_PATH = NOTEBOOK_DIR / 'saved_rois' / 'Pt_L3_roi.json'
USE_SAVED_ROI = True
OPEN_ROI_EDITOR = False
ROI_EDITOR_BACKEND = 'pyqtgraph'
ROI_EDITOR_BLOCK = True
SAVE_ROI_AFTER_APPLY = False  # keep Pt_L3_roi.json unchanged while applying it to all data

ROI_ROW_RANGE = None  # manual horizontal ROI for every run, e.g. (168, 230)
USE_TILTED_ROI = True
SHOW_ROI_PREVIEW = False
ROI_PREVIEW_FRAME = 0
TILTED_ROI_FRAME_AVERAGE = 5
TILTED_ROI_THRESHOLD_FRACTION = 0.55
TILTED_ROI_SHRINK_FRACTION = 0.90
TILTED_ROI_SMOOTH_SIGMA_ROWS = 2.0
TILTED_ROI_SMOOTH_SIGMA_COLS = 6.0

AVER_N = 5
CHUNK_SIZE = 250  # frames read per batch; WRITE_SINGLE_HTML keeps one output HTML per scan
MEDIAN_SIZE = 3
NORM_WINDOW = 80
NORM_SAMPLE_FRAMES = 100
WRITE_SINGLE_HTML = True
CLEAR_EXISTING_PREVIEW_HTML = True
ADD_ROI_OVERLAY_TO_HTML = True
ROI_OVERLAY_COLOR = 'cyan'
ROI_OVERLAY_FILL = 'rgba(0, 255, 255, 0.16)'

In [ ]:
def entry_from_selected_path(path_setting):
    if path_setting is None:
        return None
    path = Path(path_setting).expanduser()
    if not path.is_absolute():
        path = NOTEBOOK_DIR / path
    path = path.resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    return inspect_export(path)


def normalize_selected_path(path_setting):
    path = Path(path_setting).expanduser()
    if not path.is_absolute():
        path = DATA_DIR / path
    return path.resolve()


def count_multiple_hdf_paths():
    return sorted(DATA_DIR.glob(COUNT_MULTIPLE_GLOB))


def nearest_entry(target_entry, candidate_entries, prefer='nearest'):
    if not candidate_entries:
        raise ValueError('No flatfield entries are available.')
    target_time = target_entry['scan_time']
    if prefer == 'after':
        after = [entry for entry in candidate_entries if entry['scan_time'] >= target_time]
        if after:
            return min(after, key=lambda entry: entry['scan_time'] - target_time)
    elif prefer == 'before':
        before = [entry for entry in candidate_entries if entry['scan_time'] <= target_time]
        if before:
            return min(before, key=lambda entry: target_time - entry['scan_time'])
    elif prefer != 'nearest':
        raise ValueError(f'Unsupported FLATFIELD_PREFERENCE={prefer!r}.')
    return min(candidate_entries, key=lambda entry: abs((entry['scan_time'] - target_time).total_seconds()))


def entry_summary(entry):
    return {
        'name': entry['name'],
        'sample_name': entry['sample_name'],
        'scan_id': int(entry['scan_id']),
        'time': entry['scan_time'].isoformat() if entry['scan_time'] is not None else None,
        'shape': list(entry['shape']),
        'nframes': int(entry['nframes']),
        'path': str(entry['path']),
    }


all_count_multiple_paths = count_multiple_hdf_paths()
if FIXED_FLATFIELD_FILE is not None:
    flatfield_paths = [normalize_selected_path(FIXED_FLATFIELD_FILE)]
else:
    flatfield_paths = [
        path for path in all_count_multiple_paths
        if FLATFIELD_NAME_TOKEN.lower() in path.name.lower()
    ]
data_paths = [
    path for path in all_count_multiple_paths
    if not any(token.lower() in path.name.lower() for token in DATA_EXCLUDE_TOKENS)
]
if ONLY_DATA_FILES is not None:
    selected_data_paths = {normalize_selected_path(path) for path in ONLY_DATA_FILES}
    data_paths = [path for path in data_paths if path.resolve() in selected_data_paths]

batch_skipped = []
flatfield_entries = []
for path in flatfield_paths:
    try:
        flatfield_entries.append(inspect_export(path))
    except Exception as exc:
        batch_skipped.append({
            'role': 'flatfield',
            'name': path.name,
            'path': str(path),
            'status': 'skipped',
            'reason': f'{type(exc).__name__}: {exc}',
        })

if not flatfield_entries:
    raise RuntimeError(f'No usable flatfields matched {FLATFIELD_NAME_TOKEN!r} in {DATA_DIR}.')

batch_pairs = []
for path in data_paths:
    try:
        data_entry = inspect_export(path)
        if FIXED_FLATFIELD_FILE is not None:
            flat_entry = flatfield_entries[0]
        else:
            flat_entry = nearest_entry(data_entry, flatfield_entries, prefer=FLATFIELD_PREFERENCE)
        delta_minutes = abs((flat_entry['scan_time'] - data_entry['scan_time']).total_seconds()) / 60.0
        batch_pairs.append({
            'data_entry': data_entry,
            'flat_entry': flat_entry,
            'delta_minutes': float(delta_minutes),
        })
    except Exception as exc:
        batch_skipped.append({
            'role': 'data',
            'name': path.name,
            'path': str(path),
            'status': 'skipped',
            'reason': f'{type(exc).__name__}: {exc}',
        })

batch_plan = [
    {
        'data': entry_summary(pair['data_entry']),
        'flatfield': entry_summary(pair['flat_entry']),
        'delta_minutes': pair['delta_minutes'],
    }
    for pair in batch_pairs
]

batch_plan, batch_skipped

## Batch Pairing

The notebook now discovers `UFIS*count_multiple` HDF files from `DATA_DIR`, skips files with `flatfield` in their names as data inputs, and pairs each remaining data file with `FIXED_FLATFIELD_FILE` when set. If `FIXED_FLATFIELD_FILE = None`, it falls back to the nearest usable `flatfield_lambda-count_multiple` HDF.

In [ ]:
ROI_JSON_DIR = NOTEBOOK_DIR / 'saved_rois'
ROI_JSON_DIR.mkdir(exist_ok=True)
BATCH_OUTPUT_ROOT = DATA_DIR / 'preliminary_results'
BATCH_OUTPUT_ROOT.mkdir(exist_ok=True)
BATCH_SUMMARY_PATH = BATCH_OUTPUT_ROOT / 'batch_preview_summary.json'


def batch_roi_json_path(data_entry):
    if ROI_JSON_PATH is not None:
        return Path(ROI_JSON_PATH).expanduser().resolve()
    if ROI_JSON_NAME is not None:
        return ROI_JSON_DIR / ROI_JSON_NAME
    return ROI_JSON_DIR / f"{preview_stem(data_entry)}_roi.json"


def prepare_batch_paths(data_entry, flat_entry):
    source_flat_path = resolve_uid_path(flat_entry)
    source_data_path = resolve_uid_path(data_entry)
    flat_path = ensure_legacy_lambda_view(
        source_flat_path,
        COMPAT_DATA_DIR / f"{preview_stem(flat_entry)}.h5",
    )
    data_path = ensure_legacy_lambda_view(
        source_data_path,
        COMPAT_DATA_DIR / f"{preview_stem(data_entry)}.h5",
    )
    output_dir = BATCH_OUTPUT_ROOT / f"preliminary_results_{preview_stem(data_entry)}"
    roi_json_path = batch_roi_json_path(data_entry)
    return source_flat_path, source_data_path, flat_path, data_path, output_dir, roi_json_path


def initial_roi_for_paths(data_path, flat_path, roi_json_path):
    if USE_SAVED_ROI and roi_json_path.exists():
        return load_roi_json(roi_json_path)
    if ROI_ROW_RANGE is not None:
        return {
            'kind': 'row_range',
            'row_start': int(ROI_ROW_RANGE[0]),
            'row_stop': int(ROI_ROW_RANGE[1]),
            'row_bounds': [int(ROI_ROW_RANGE[0]), int(ROI_ROW_RANGE[1])],
        }
    if USE_TILTED_ROI:
        return infer_tilted_band_roi_from_paths(
            data_path=str(data_path),
            flat_path=str(flat_path),
            frame_index=ROI_PREVIEW_FRAME,
            frame_average=TILTED_ROI_FRAME_AVERAGE,
            threshold_fraction=TILTED_ROI_THRESHOLD_FRACTION,
            shrink_fraction=TILTED_ROI_SHRINK_FRACTION,
            smooth_sigma_rows=TILTED_ROI_SMOOTH_SIGMA_ROWS,
            smooth_sigma_cols=TILTED_ROI_SMOOTH_SIGMA_COLS,
            median_size=MEDIAN_SIZE,
        )
    raise ValueError('Set ROI_ROW_RANGE, enable USE_TILTED_ROI, or load a saved ROI JSON.')


def roi_metadata_for_pair(data_entry, flat_entry, data_path, flat_path, source_data_path, source_flat_path):
    return {
        'data_date': data_entry['scan_time'].strftime('%Y-%m-%d'),
        'flat_date': flat_entry['scan_time'].strftime('%Y-%m-%d'),
        'data_scan_id': int(data_entry['scan_id']),
        'flat_scan_id': int(flat_entry['scan_id']),
        'data_sample_name': data_entry['sample_name'],
        'flat_sample_name': flat_entry['sample_name'],
        'data_path': str(data_path),
        'flat_path': str(flat_path),
        'source_data_path': str(source_data_path),
        'source_flat_path': str(source_flat_path),
        'preview_frame': int(ROI_PREVIEW_FRAME),
    }


def open_roi_selector_for_pair(pair_index=0, save_selected=True):
    if not batch_pairs:
        raise RuntimeError('No batch pairs are available. Run the batch pairing cell first.')
    pair = batch_pairs[int(pair_index)]
    data_entry = pair['data_entry']
    flat_entry = pair['flat_entry']
    source_flat_path, source_data_path, flat_path, data_path, _output_dir, roi_json_path = prepare_batch_paths(data_entry, flat_entry)
    selected_roi = initial_roi_for_paths(data_path, flat_path, roi_json_path)
    selected_roi = select_roi_for_processing(
        data_path=data_path,
        flat_path=flat_path,
        initial_roi=selected_roi,
        roi_json_path=roi_json_path,
        data_entry=data_entry,
    )
    if save_selected:
        save_roi_json(
            roi_json_path,
            selected_roi,
            metadata=roi_metadata_for_pair(data_entry, flat_entry, data_path, flat_path, source_data_path, source_flat_path),
        )
    return selected_roi


def html_files_for_output(output_dir):
    return sorted(str(path) for path in Path(output_dir).glob('*.html'))


def clear_preview_html_files(output_dir):
    if not CLEAR_EXISTING_PREVIEW_HTML:
        return []
    removed = []
    for html_path in sorted(Path(output_dir).glob('preview_*.html')):
        html_path.unlink()
        removed.append(str(html_path))
    return removed


def detector_image_shape(path):
    with h5py.File(path, 'r') as h5_file:
        shape = detector_dataset(h5_file).shape
    if len(shape) < 2:
        raise ValueError(f'Expected detector images in {path}, got shape={shape!r}.')
    return int(shape[-2]), int(shape[-1])


def roi_overlay_trace_specs(image_shape, row_range, roi, axes=(1, 2, 3)):
    cols, top_rows, bottom_rows = roi_boundary_rows(image_shape, row_range=row_range, roi=roi)
    x = [int(v) for v in cols]
    top = [float(v) for v in top_rows]
    bottom = [float(v) for v in bottom_rows]
    band_x = x + list(reversed(x))
    band_y = top + list(reversed(bottom))
    traces = []
    for axis_number in axes:
        axis_suffix = '' if axis_number == 1 else str(axis_number)
        axis_ref = {
            'xaxis': f'x{axis_suffix}',
            'yaxis': f'y{axis_suffix}',
        }
        show_legend = axis_number == max(axes)
        traces.append({
            'type': 'scatter',
            'mode': 'lines',
            'x': band_x,
            'y': band_y,
            'fill': 'toself',
            'fillcolor': ROI_OVERLAY_FILL,
            'line': {'color': ROI_OVERLAY_COLOR, 'width': 0.5},
            'name': 'ROI band',
            'hoverinfo': 'skip',
            'showlegend': bool(show_legend),
            **axis_ref,
        })
        for label, y_values in (('ROI top', top), ('ROI bottom', bottom)):
            traces.append({
                'type': 'scatter',
                'mode': 'lines',
                'x': x,
                'y': y_values,
                'line': {'color': ROI_OVERLAY_COLOR, 'width': 2},
                'name': label,
                'hoverinfo': 'skip',
                'showlegend': False,
                **axis_ref,
            })
    return traces


def inject_roi_into_existing_snapshot(html_text, image_shape, row_range, roi):
    marker = 'const snapshotGd = document.getElementById("dxas_snapshot_plot");'
    if marker not in html_text or 'dxas-roi-overlay-start' in html_text:
        return html_text, False
    traces = roi_overlay_trace_specs(image_shape, row_range=row_range, roi=roi, axes=(1, 2, 3))
    overlay_js = f'''/* dxas-roi-overlay-start */
    const dxasRoiOverlayTraces = {json.dumps(traces)};
    snapshotFig.data = (snapshotFig.data || []).concat(dxasRoiOverlayTraces);
    snapshotFig.layout = snapshotFig.layout || {{}};
    snapshotFig.layout.showlegend = true;
    /* dxas-roi-overlay-end */
    {marker}'''
    html_text = html_text.replace(marker, overlay_js, 1)
    roi_note = '<div><b>ROI:</b> cyan band marks the detector rows used to generate the spectra.</div>'
    mux_note = '<div><b>Mux frame:</b> computed from data frame'
    if roi_note not in html_text and mux_note in html_text:
        html_text = html_text.replace(mux_note, roi_note + '\n      ' + mux_note, 1)
    return html_text, True


def parse_chunk_start_from_html_name(html_path):
    match = re.search(r'preview_(\d+)-\d+_N\d+\.html$', Path(html_path).name)
    return int(match.group(1)) if match else int(ROI_PREVIEW_FRAME)


def mux_snapshot_figure_json(data_path, flat_path, row_range, roi, frame_index, title):
    import plotly.graph_objects as go

    mux_preview, preview_frame_index = load_mux_preview_frame(data_path, flat_path, frame_index)
    finite = mux_preview[np.isfinite(mux_preview)]
    if finite.size:
        vmin, vmax = np.nanpercentile(finite, [1, 99])
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
            vmin = float(np.nanmin(finite))
            vmax = float(np.nanmax(finite))
        if vmax <= vmin:
            vmax = vmin + 1.0
    else:
        vmin, vmax = 0.0, 1.0

    fig = go.Figure()
    fig.add_trace(
        go.Heatmap(
            z=np.asarray(mux_preview, dtype=float),
            colorscale='magma',
            zmin=float(vmin),
            zmax=float(vmax),
            colorbar=dict(title='Mux', thickness=8),
            hovertemplate='x=%{x}<br>y=%{y}<br>mux=%{z}<extra></extra>',
        )
    )
    for trace in roi_overlay_trace_specs(mux_preview.shape, row_range=row_range, roi=roi, axes=(1,)):
        trace = dict(trace)
        trace.pop('xaxis', None)
        trace.pop('yaxis', None)
        fig.add_trace(go.Scatter(**trace))
    fig.update_layout(
        title=f'{title}<br><sup>Mux frame {preview_frame_index}; cyan band is the ROI used for spectra</sup>',
        height=620,
        autosize=True,
        font=dict(size=11),
        margin=dict(t=85, b=55, l=70, r=80),
        showlegend=True,
    )
    fig.update_xaxes(title_text='Detector x pixel')
    fig.update_yaxes(title_text='Detector y pixel')
    return fig.to_json()


def flatfield_link_html(html_path, flat_path, source_flat_path=None):
    flatfield_path = Path(source_flat_path or flat_path)
    try:
        href = os.path.relpath(flatfield_path, start=Path(html_path).parent)
    except ValueError:
        href = str(flatfield_path)
    href = quote(href.replace(os.sep, '/'), safe='/._-()%')
    label = flatfield_path.name
    return f'<div><b>Flatfield:</b> <a href="{html.escape(href, quote=True)}">{html.escape(label)}</a></div>'


def ensure_flatfield_link_text(html_text, html_path, flat_path, source_flat_path=None):
    if '<b>Flatfield:</b>' in html_text:
        return html_text, False
    if '<b>Flat file:</b>' in html_text:
        return html_text.replace('<b>Flat file:</b>', '<b>Flatfield:</b>', 1), True
    flatfield_line = '      ' + flatfield_link_html(html_path, flat_path, source_flat_path=source_flat_path) + '\n'
    for marker in (
        '      <div><b>Snapshot:</b> mux image from the first frame in this HTML chunk.</div>\n',
        '      <div><b>Mux frame:</b> computed from data frame',
    ):
        if marker in html_text:
            if marker.endswith('\n'):
                return html_text.replace(marker, marker + flatfield_line, 1), True
            insert_at = html_text.find('</div>', html_text.find(marker))
            if insert_at >= 0:
                insert_at += len('</div>')
                return html_text[:insert_at] + '\n' + flatfield_line.rstrip('\n') + html_text[insert_at:], True
    return html_text, False


def append_roi_snapshot_section(html_text, html_path, data_path, flat_path, row_range, roi, data_entry, source_flat_path=None):
    if 'dxas-roi-overlay-start' in html_text:
        return ensure_flatfield_link_text(html_text, html_path, flat_path, source_flat_path=source_flat_path)
    frame_index = parse_chunk_start_from_html_name(html_path)
    div_id = 'dxas_roi_snapshot_plot_' + re.sub(r'[^0-9A-Za-z_]+', '_', Path(html_path).stem)
    fig_json = mux_snapshot_figure_json(
        data_path,
        flat_path,
        row_range=row_range,
        roi=roi,
        frame_index=frame_index,
        title=f"ROI snapshot - {data_entry['name']}",
    )
    flatfield_line = flatfield_link_html(html_path, flat_path, source_flat_path=source_flat_path)
    section = f'''
    <!-- dxas-roi-overlay-start -->
    <div class="snapshot-links">
      <div><b>ROI:</b> cyan band marks the detector rows used to generate the spectra.</div>
      <div><b>Snapshot:</b> mux image from the first frame in this HTML chunk.</div>
      {flatfield_line}
    </div>
    <div id="{div_id}" style="width:100%; margin-top: 16px;"></div>
    <!-- dxas-roi-overlay-end -->
'''
    controls_marker = '    <div id="controls">'
    if controls_marker in html_text:
        html_text = html_text.replace(controls_marker, section + controls_marker, 1)
    else:
        html_text = html_text.replace('</body>', section + '</body>', 1)
    script = f'''
  <script>
    (function() {{
      const roiSnapshotFig = {fig_json};
      const roiSnapshotDiv = document.getElementById("{div_id}");
      if (roiSnapshotDiv && window.Plotly) {{
        Plotly.newPlot(roiSnapshotDiv, roiSnapshotFig.data, roiSnapshotFig.layout, {{responsive: true, displaylogo: false}});
      }}
    }})();
  </script>
'''
    html_text = html_text.replace('</body>', script + '</body>', 1)
    return html_text, True


def add_roi_overlay_to_html_files(output_dir, data_path, flat_path, row_range, roi, data_entry, source_flat_path=None):
    if not ADD_ROI_OVERLAY_TO_HTML:
        return []
    image_shape = detector_image_shape(data_path)
    updated = []
    for html_path in sorted(Path(output_dir).glob('*.html')):
        html_text = html_path.read_text(encoding='utf-8')
        if 'dxas_snapshot_plot' in html_text and 'snapshotFig' in html_text:
            new_text, changed = inject_roi_into_existing_snapshot(html_text, image_shape, row_range=row_range, roi=roi)
        else:
            new_text, changed = append_roi_snapshot_section(
                html_text,
                html_path,
                data_path=data_path,
                flat_path=flat_path,
                row_range=row_range,
                roi=roi,
                data_entry=data_entry,
                source_flat_path=source_flat_path,
            )
        new_text, flatfield_changed = ensure_flatfield_link_text(
            new_text,
            html_path,
            flat_path=flat_path,
            source_flat_path=source_flat_path,
        )
        changed = changed or flatfield_changed
        if changed:
            html_path.write_text(new_text, encoding='utf-8')
            updated.append(str(html_path))
    return updated

## Select ROI Once

Run this cell to open the PyQtGraph selector for the first planned UFIS scan, save `saved_rois/Pt_L3_roi.json`, and reuse that ROI for all batch exports.

In [ ]:
selected_roi_for_batch = open_roi_selector_for_pair(pair_index=0, save_selected=True)
selected_roi_for_batch

## Batch Export

The next cell processes every planned pair using `saved_rois/Pt_L3_roi.json`. Run the one-time selector cell above before exporting if you want to revise the ROI; ROI highlighting is added to each output HTML file.

In [ ]:
def export_preview_for_pair(pair, index=None, total=None):
    data_entry = pair['data_entry']
    flat_entry = pair['flat_entry']
    prefix = f"[{index}/{total}] " if index is not None and total is not None else ''
    result = {
        'data': entry_summary(data_entry),
        'flatfield': entry_summary(flat_entry),
        'delta_minutes': pair['delta_minutes'],
        'status': 'started',
    }
    try:
        print(f"{prefix}Processing {data_entry['name']} with flatfield {flat_entry['name']}")
        source_flat_path, source_data_path, flat_path, data_path, output_dir, roi_json_path = prepare_batch_paths(data_entry, flat_entry)
        selected_roi = initial_roi_for_paths(data_path, flat_path, roi_json_path)
        if OPEN_ROI_EDITOR:
            selected_roi = select_roi_for_processing(
                data_path=data_path,
                flat_path=flat_path,
                initial_roi=selected_roi,
                roi_json_path=roi_json_path,
                data_entry=data_entry,
            )

        roi = None
        if selected_roi is not None:
            row_range = tuple(int(v) for v in selected_roi['row_bounds'])
            if selected_roi.get('kind') == 'tilted_band':
                roi = selected_roi
        else:
            row_range = tuple(int(v) for v in ROI_ROW_RANGE)

        if SAVE_ROI_AFTER_APPLY and selected_roi is not None:
            save_roi_json(
                roi_json_path,
                selected_roi,
                metadata={
                    'data_date': data_entry['scan_time'].strftime('%Y-%m-%d'),
                    'flat_date': flat_entry['scan_time'].strftime('%Y-%m-%d'),
                    'data_scan_id': int(data_entry['scan_id']),
                    'flat_scan_id': int(flat_entry['scan_id']),
                    'data_sample_name': data_entry['sample_name'],
                    'flat_sample_name': flat_entry['sample_name'],
                    'data_path': str(data_path),
                    'flat_path': str(flat_path),
                    'source_data_path': str(source_data_path),
                    'source_flat_path': str(source_flat_path),
                    'preview_frame': int(ROI_PREVIEW_FRAME),
                },
            )

        norm_range = infer_norm_range(
            data_path,
            flat_path,
            row_range=row_range,
            roi=roi,
            window=NORM_WINDOW,
            sample_frames=NORM_SAMPLE_FRAMES,
        )

        if SHOW_ROI_PREVIEW:
            show_roi_preview(data_path, flat_path, row_range=row_range, roi=roi, frame_index=ROI_PREVIEW_FRAME)

        preview_kwargs = dict(
            data_path=str(data_path),
            flat_path=str(flat_path),
            aver_n=AVER_N,
            flat_range=row_range,
            roi=roi,
            norm_x1=norm_range[0],
            norm_x2=norm_range[1],
            chunk_size=CHUNK_SIZE,
            cmap_name='magma',
            display_inline=False,
            median_size=MEDIAN_SIZE,
            output_dir=str(output_dir),
        )

        output_dir.mkdir(parents=True, exist_ok=True)
        removed_preview_html_files = clear_preview_html_files(output_dir)
        full_preview_written = False
        if WRITE_SINGLE_HTML:
            full_preview_kwargs = dict(
                preview_kwargs,
                snapshot_frame=ROI_PREVIEW_FRAME,
                flat_snapshot_frame=0,
                mux_snapshot_frame=ROI_PREVIEW_FRAME,
                data_label=source_data_path.name,
                flat_label=source_flat_path.name,
                data_link=str(source_data_path),
                flat_link=str(source_flat_path),
            )
            preview_spectra_html(**full_preview_kwargs)
            full_preview_written = True
        else:
            plot_spectra_in_chunks(
                **preview_kwargs,
                x1=norm_range[0],
                x2=400,
                output_format='html',
            )

        roi_overlay_html_files = add_roi_overlay_to_html_files(
            output_dir,
            data_path=data_path,
            flat_path=flat_path,
            row_range=row_range,
            roi=roi,
            data_entry=data_entry,
            source_flat_path=source_flat_path,
        )

        result.update({
            'status': 'completed',
            'row_range': list(row_range),
            'norm_range': list(norm_range),
            'roi_json_path': str(roi_json_path),
            'source_data_path': str(source_data_path),
            'source_flat_path': str(source_flat_path),
            'compat_data_path': str(data_path),
            'compat_flat_path': str(flat_path),
            'output_dir': str(output_dir),
            'html_files': html_files_for_output(output_dir),
            'roi_overlay_html_files': roi_overlay_html_files,
            'full_preview_written': bool(full_preview_written),
            'removed_preview_html_files': removed_preview_html_files,
        })
        print(f"{prefix}Completed {data_entry['name']} -> {output_dir}")
    except Exception as exc:
        result.update({
            'status': 'failed',
            'reason': f'{type(exc).__name__}: {exc}',
        })
        print(f"{prefix}Failed {data_entry['name']}: {type(exc).__name__}: {exc}")
    return result


batch_results = []
for index, pair in enumerate(batch_pairs, start=1):
    batch_results.append(export_preview_for_pair(pair, index=index, total=len(batch_pairs)))

batch_results

In [ ]:
batch_summary = {
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'data_dir': str(DATA_DIR),
    'settings': {
        'count_multiple_glob': COUNT_MULTIPLE_GLOB,
        'flatfield_name_token': FLATFIELD_NAME_TOKEN,
        'fixed_flatfield_file': FIXED_FLATFIELD_FILE,
        'data_exclude_tokens': list(DATA_EXCLUDE_TOKENS),
        'only_data_files': ONLY_DATA_FILES,
        'flatfield_preference': FLATFIELD_PREFERENCE,
        'roi_json_path': str(Path(ROI_JSON_PATH).expanduser().resolve()) if ROI_JSON_PATH is not None else None,
        'use_saved_roi': bool(USE_SAVED_ROI),
        'open_roi_editor': bool(OPEN_ROI_EDITOR),
        'roi_editor_backend': ROI_EDITOR_BACKEND,
        'roi_editor_block': bool(ROI_EDITOR_BLOCK),
        'show_roi_preview': bool(SHOW_ROI_PREVIEW),
        'save_roi_after_apply': bool(SAVE_ROI_AFTER_APPLY),
        'aver_n': int(AVER_N),
        'chunk_size': int(CHUNK_SIZE),
        'median_size': int(MEDIAN_SIZE),
        'norm_window': int(NORM_WINDOW),
        'norm_sample_frames': int(NORM_SAMPLE_FRAMES),
        'write_single_html': bool(WRITE_SINGLE_HTML),
        'clear_existing_preview_html': bool(CLEAR_EXISTING_PREVIEW_HTML),
        'add_roi_overlay_to_html': bool(ADD_ROI_OVERLAY_TO_HTML),
    },
    'planned_pairs': batch_plan,
    'discovery_skipped': batch_skipped,
    'results': batch_results,
}
BATCH_SUMMARY_PATH.write_text(json.dumps(batch_summary, indent=2), encoding='utf-8')

completed_results = [result for result in batch_results if result['status'] == 'completed']
failed_results = [result for result in batch_results if result['status'] != 'completed']

BATCH_SUMMARY_PATH, len(completed_results), len(failed_results), len(batch_skipped)

In [ ]:
print(f'Batch summary: {BATCH_SUMMARY_PATH}')
print(f'ROI JSON: {Path(ROI_JSON_PATH).expanduser().resolve()}')
print(f'Completed: {len(completed_results)}')
print(f'Failed during export: {len(failed_results)}')
print(f'Skipped during discovery: {len(batch_skipped)}')
print(f'ROI-overlayed HTML files: {sum(len(result.get("roi_overlay_html_files", [])) for result in completed_results)}')

for result in completed_results:
    print(f"OK: {result['data']['name']} -> {result['output_dir']}")
    for html_file in result['html_files']:
        print(f"  {html_file}")

for result in failed_results:
    print(f"FAILED: {result['data']['name']} | {result.get('reason', '')}")

for item in batch_skipped:
    print(f"SKIPPED: {item['name']} | {item['reason']}")

completed_results, failed_results, batch_skipped